<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 9 · Interest-Rate Swaps

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch09_interest_rate_swaps.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

An interest-rate swap is two legs valued on two curves: one curve discounts, another projects floating coupons. In this lab you compute a multi-curve par rate from projected annual forwards, value an off-market receive-fixed swap against that par rate, and check the receiver DV01 sign when the par rate rises one basis point. You run the two controls every valuation sheet needs, par repricing and a signed sensitivity check, and you state which curve each number depends on and which risks a fixed-discount DV01 leaves out.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The script projects five annual floating coupons from the supplied forwards, discounts them on a separate discount curve, and divides by the fixed-leg annuity to print a par rate of 0.0407. It then values a 100 million USD receive-fixed swap at a 4.0% contract rate at -308,100 USD and reports a receiver DV01 magnitude of 44,390 USD per basis point, holding discount factors fixed to isolate projection-curve risk.

In [ ]:
SCRIPT = ROOT / "code" / "ch09_interest_rate_swaps.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

1. Verify the annuity: the five annual accrual-discount products sum to 4.439, and the par rate is the discounted floating stream divided by it, 0.180641 / 4.439 = 0.0407.
2. Confirm the value sign: the 4.0% contract rate sits below the 4.0694% par rate, so the receive-fixed swap is worth -308,100 USD on 100 million notional.
3. Check the DV01 arithmetic, 100,000,000 * 4.439 * 0.0001 = 44,390 USD per basis point, and note it is a positive magnitude for a receiver who loses when the par rate rises.
4. Note the conventions: annual payments with accrual fractions of one, discounting separated from forward projection, and discount factors held fixed in the DV01, so a joint curve shock would give a different number.
5. Treat the quote table as an illustrative teaching snapshot rather than a market curve.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch09_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Chapter-specific checks

Recompute the par rate, verify par repricing, and show the receiver/payer DV01 sign convention explicitly.

In [ ]:
swap = pd.read_csv(DATA / "ch09_swap_inputs.csv")
annuity = (swap.alpha * swap.discount).sum()
floating = (swap.alpha * swap.discount * swap.forward).sum()
par_rate = floating / annuity
notional = 50_000_000
receiver_value_at_par = notional * annuity * (par_rate - par_rate)
receiver_dv01 = -notional * annuity * 0.0001
payer_dv01 = -receiver_dv01
pd.DataFrame({
    "check": ["par_rate", "par_repricing", "receiver_dv01", "payer_dv01"],
    "value": [par_rate, receiver_value_at_par, receiver_dv01, payer_dv01],
}).round(6)

## Mini-lab: Price a fixed leg from bootstrapped discounts

The fixed leg of a swap is an annuity: notional times fixed rate times the sum of the discount factors on its payment dates. This lab bootstraps three annual discount factors from illustrative par swap quotes, then prices the 4.0% fixed leg of a three-year swap on 100 million notional. A closing identity confirms the same annuity reproduces the input par rate.

In [ ]:
import numpy as np

par_rates = np.array([0.036, 0.038, 0.0395])  # annual par swap rates
discounts = []  # bootstrapped discount factors
for rate in par_rates:
    known_pv = sum(discounts)  # earlier fixed coupons
    discounts.append((1 - rate * known_pv) / (1 + rate))
discounts = np.array(discounts)

annuity = discounts.sum()  # fixed-leg annuity (annual payments)
notional = 100_000_000  # swap notional (USD)
fixed_pv = notional * 0.04 * annuity  # 4 percent fixed-leg PV
print("discounts:", discounts.round(6))
print("annuity:", round(annuity, 6))
print("fixed-leg PV (USD):", round(fixed_pv, 2))

### What to observe

The bootstrapped discounts are 0.965251, 0.928054, and 0.890057, summing to an annuity of 2.783363. The 4.0% fixed leg on 100 million notional is therefore worth 11,133,450.16 USD, and (1 - 0.890057) / 2.783363 returns exactly 3.95%, the final input par rate.

## Exercises

1. **Reprice the par swap.** Set the contract fixed rate equal to the printed par rate and confirm the receive-fixed value is zero. *Hint:* value is notional times annuity times the rate difference.
2. **Value the payer side.** State the payer swap's value and the sign of its DV01 for a one-basis-point par-rate rise. *Hint:* the payer value is the negative of the receiver value.
3. **Change the fixed rate.** Value the receive-fixed swap at 4.5% instead of 4.0% on the same inputs. *Hint:* only the rate difference times the annuity moves the value.
4. **Bump the discount curve.** Recompute the par rate after a one-basis-point shift applied through the discount factors and explain the direction of the move. *Hint:* multiply each discount factor by exp(-0.0001 * t).
5. **Scope the DV01.** Explain which risk the printed 44,390 USD figure measures and which it excludes. *Hint:* discount factors are held fixed, so only projection-curve risk is captured.
6. **Write the controls.** List the fields a reviewer needs before signing off on a swap mark. *Hint:* the chapter's controls section enumerates payer or receiver direction, curves, notional, and sensitivity sign.

For more detail, see Chapter 9 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">